# Reto 2 — Visión: identificación, ángulo y recuperación en COIL-100

Notebook de punta a punta y reproducible: **adquisición de datos → EDA → experimentación por protocolo →
selección de modelo → análisis de errores → exportación del artefacto**. Toda la lógica vive en `src/vision/`;
aquí solo se orquesta y se muestran resultados. Semilla global 42. Los experimentos quedan registrados en
MLflow (`sqlite:///mlflow.db`, experimento `vision-coil100`).

**Regla de oro metodológica:** todo hiperparámetro y todo umbral se elige con particiones de *validación*
construidas exclusivamente con las vistas de entrenamiento de cada protocolo; el test se evalúa una sola vez.

In [1]:
import sys, json, time, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.vision import data as vdata, analysis as va, experiments as vexp
from src.vision.data import ensure_dataset, sha256_file, data_version_hash, protocol_split, validation_split, COIL_SHA256
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
np.random.seed(42)
T0 = time.time()

## 1. Adquisición y verificación de datos

Si `data/raw/coil-100/` no existe, se descarga el zip oficial
(`https://www.cs.columbia.edu/CAVE/databases/SLAM_coil-20_coil-100/coil-100/coil-100.zip`), se verifica su
SHA-256 y se extrae. Además se calcula un **hash de versión de datos** (SHA-256 sobre nombre + contenido de
cada PNG) que viaja en MLflow y en `metadata.json` del artefacto.

In [2]:
img_dir = ensure_dataset()
zip_path = img_dir.parent / "coil-100.zip"
if zip_path.exists():
    print("SHA-256 zip:", sha256_file(zip_path), "OK" if sha256_file(zip_path) == COIL_SHA256 else "DISTINTO")
DATA_HASH = data_version_hash()
print("imágenes:", len(list(img_dir.glob("obj*__*.png"))), "| hash de versión de datos:", DATA_HASH[:16], "…")
data = vexp.load_all()          # imágenes + características clásicas + píxeles crudos + embeddings ResNet18 (cache en data/runtime)
ds = data["ds"]
print(ds.images.shape, "clases:", len(np.unique(ds.labels)), "ángulos:", np.unique(ds.angles)[:4], "…", np.unique(ds.angles)[-1])

SHA-256 zip: fcb4b8b01c0778f416d8d7a09f45082c4f7075cc74743dcddd5c44307bd4afe7 OK


imágenes: 7200 | hash de versión de datos: e9e73124fbf0e964 …


(7200, 128, 128, 3) clases: 100 ángulos: [ 0  5 10 15] … 355


## 2. Análisis exploratorio (EDA)

In [3]:
fig = va.sample_grid(ds); va.savefig(fig, "01_sample_grid"); plt.show()
st = va.image_stats(ds)
desc = pd.DataFrame({k: pd.Series(v).describe(percentiles=[.001, .01, .5, .99, .999]) for k, v in st.items()}).T
display(desc.round(3))

,count,mean,std,min,0.1%,1%,50%,99%,99.9%,max
brightness,7200.0,0.268,0.071,0.129,0.134,0.151,0.265,0.440,0.491,0.526
object_fraction,7200.0,0.471,0.147,0.128,0.131,0.185,0.468,0.781,0.930,0.952
background_median,7200.0,26.297,2.154,22.000,23.000,24.000,26.000,35.000,43.000,47.000
sharpness,7200.0,327.561,251.640,44.245,46.601,54.948,255.630,1282.385,1810.964,2041.270


**Lectura.** 100 objetos × 72 vistas (paso exacto de 5°), 128×128 RGB, **fondo oscuro uniforme** (mediana del
borde ≈ 25/255, *no* 0) e **iluminación controlada**: brillo y nitidez varían poco. Consecuencias de diseño:

* El problema de clase es "fácil" en laboratorio; el riesgo real está en el ángulo y en la extrapolación a planta.
* Como el fondo es casi constante, una máscara por umbral de gris (> 40) separa objeto/fondo; en planta esto no
  vale (cinta, reflejos) → compuerta de calidad + segmentación robusta serían requisitos.
* Las estadísticas de brillo/nitidez/fracción de objeto definen los umbrales de la **compuerta de calidad**
  determinista (`src/vision/quality.py`, percentil extremo × margen).

In [4]:
from src.vision.models import VisionModel
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
Zall = PCA(150, random_state=42).fit_transform(StandardScaler().fit_transform(data["classic"]))  # solo para EDA
sym = va.symmetry_analysis(Zall, ds)
fig = va.symmetry_figure(sym); va.savefig(fig, "02_symmetry"); plt.show()
symdf = pd.DataFrame({"perfil": sym["profile_score"], "loo": sym["loo_score"], "d180/d90": sym["ratio_180_90"]})
symdf.index = [f"obj{o}" for o in symdf.index]
print("Más rotacionalmente ambiguos según el perfil (d@5° / d@≥45°):")
display(symdf.sort_values("perfil", ascending=False).head(10).round(3))
print("Vista 'gemela' más parecida a ≥30° (índice LOO):")
display(symdf.sort_values("loo", ascending=False).head(8).round(3))
print(f"Objetos con vistas opuestas (180°) más parecidas que a 90°: {(symdf['d180/d90'] < 1).sum()} de 100")

Más rotacionalmente ambiguos según el perfil (d@5° / d@≥45°):


,perfil,loo,d180/d90
obj95,0.768,0.097,1.069
obj94,0.593,0.042,1.061
obj47,0.591,0.028,1.094
obj25,0.552,0.000,0.977
obj50,0.528,0.000,0.850
obj34,0.493,0.000,1.068
obj7,0.484,0.000,1.120
obj86,0.484,0.000,0.997
obj30,0.484,0.000,0.767
obj56,0.468,0.000,1.142


Vista 'gemela' más parecida a ≥30° (índice LOO):


,perfil,loo,d180/d90
obj67,0.358,0.472,0.276
obj98,0.365,0.403,0.294
obj84,0.351,0.347,0.294
obj12,0.321,0.236,0.311
obj1,0.383,0.194,0.416
obj51,0.275,0.153,0.321
obj65,0.292,0.111,0.482
obj96,0.387,0.111,0.504


Objetos con vistas opuestas (180°) más parecidas que a 90°: 75 de 100


**Simetría rotacional (cuantificada).** Para cada objeto se mide la distancia media en el espacio de
características entre pares de vistas en función de su separación angular. En un objeto asimétrico la curva
crece con la separación; en uno simétrico (latas, tazas, donas, frascos: p. ej. obj95, obj94, obj47, obj25)
es casi plana → **el ángulo es intrínsecamente ambiguo**. El índice LOO (¿la vista más parecida está a ≥30°?)
detecta además objetos con vistas a 180° casi idénticas (cajas/frascos con dos caras parecidas).

**Tratamiento:** (i) a nivel de instancia, `estimate_angle` marca `ambiguous` si existe una vista de referencia
a ≥30° del ángulo estimado con distancia ≤ 1.10 × la mejor; (ii) a nivel de clase, el modelo productivo
(referencias cada 5°) marca como simétricas las clases con índice LOO ≥ 0.25; (iii) **empíricamente**: en
cada protocolo, las clases cuyo ángulo falla (> ±10°) en más del 25 % de sus vistas de *validación* se marcan
como de ángulo poco fiable (detecta las donas/tazas obj47, obj94, obj95 sin mirar el test); (iv) todo esto genera
`rotational_ambiguity` → **REVIEW** (el robot recibe HOLD).

**Vistas a 180°:** en ~3/4 de los objetos la vista opuesta (180°) está más cerca en el espacio de
características que la vista a 90° (la silueta de un objeto visto desde lados opuestos es casi la imagen espejo).
Por eso el ángulo no se estima con una media circular de todos los vecinos (se cancelarían modos opuestos), sino
con el modo del vecino más cercano refinado localmente, y la ambigüedad se marca explícitamente. Alternativa documentada: para un objeto
perfectamente simétrico el ángulo no importa para el agarre y podría reportarse "módulo simetría"; no lo
hacemos por defecto porque la etiqueta/código de barras sí puede depender del ángulo.

In [5]:
pairs = va.confusable_pairs(Zall, ds, top=10)
display(pd.DataFrame(pairs).round(2))

,a,b,min_inter_distance,mean_intra_nn_distance
0,obj61,obj92,10.99,7.45
1,obj29,obj32,15.77,9.11
2,obj47,obj94,17.94,5.06
3,obj67,obj84,19.00,14.58
4,obj15,obj91,19.25,13.88
5,obj1,obj15,19.81,13.35
6,obj62,obj93,19.81,9.74
7,obj15,obj69,20.23,14.47
8,obj15,obj76,20.71,14.01
9,obj21,obj44,20.84,16.15


**Clases casi duplicadas.** Los pares con menor distancia inter-objeto (comparada con la distancia típica entre
vistas adyacentes del mismo objeto) son los candidatos a confusión: mismas formas con distinto color/texto
(p. ej. cajas o latas de la misma familia). Esto motiva incluir **color** (HSV) además de forma (HOG).

**Laboratorio vs. planta.** COIL-100 tiene fondo negro, luz controlada, cámara fija a la misma distancia,
un único eje de rotación (tornamesa), sin oclusión ni desenfoque por movimiento. En planta habrá: cinta con
textura y reflejos, iluminación variable, oclusión parcial/objetos múltiples, movimiento de la cinta (blur),
pose de cámara y escala variables, empaques nuevos. Mitigaciones: (1) aumentación de datos (brillo/contraste,
blur, ruido, escala, recortes) y captura de vistas reales por SKU en la estación; (2) compuerta de calidad
(oscuridad, saturación, nitidez, cuadro vacío) antes del modelo; (3) re-calibrar umbrales con datos de planta
(los umbrales son configurables en caliente vía `/v1/admin/thresholds`); (4) **piloto en sombra** con revisión
humana del 100% antes de habilitar ACT, y monitoreo de la tasa ACT/REVIEW/REJECT y de la deriva de distancias.

## 3. Protocolos y esquema de validación

| Protocolo | Train (por objeto) | Test | Validación (solo dentro de train) |
|---|---|---|---|
| A Interpolación | 0,10,…,350 (36) | 5,15,…,355 (36) | fit paso 20°, val el resto (brecha 10° > 5° del test) |
| B Sector ciego | 60 vistas | arco contiguo de 60° (12), inicio aleatorio por objeto (semilla 42) | se oculta otro arco de 60° dentro de train (semilla 43) |
| C Escasez | 1 cada 30° (12) | las otras 60 | fit paso 60°, val el resto |
| D Desconocido | 90 objetos (paso 10°) | vistas impares de los 90 + **todas** las de 10 objetos desconocidos | 10 pseudo-desconocidos entre los 90 (semilla 43) |

Notas: en D entrenamos con las vistas de paso 10° de los 90 conocidos (no todas) para poder medir *ambos lados*
(conocidos aceptados y desconocidos rechazados) con vistas no vistas. La validación imita (y suele exagerar) la
brecha de cada protocolo; en C la brecha de validación (30°) es mayor que la del test (≤15°), por lo que el
umbral angular no se puede afinar ahí y se usa el prior 5° (= mitad de la tolerancia de ±10°).

In [6]:
rows = []
for p in "ABCD":
    sp = protocol_split(ds.labels, ds.angles, p); vs = validation_split(ds.labels, ds.angles, p)
    rows.append({"protocolo": p, "train": len(sp["train"]), "test": len(sp["test"]), "fit (val)": len(vs["fit"]),
                 "val": len(vs["val"]), "fuga train∩test": len(np.intersect1d(sp["train"], sp["test"]))})
display(pd.DataFrame(rows))
spB = protocol_split(ds.labels, ds.angles, "B"); spD = protocol_split(ds.labels, ds.angles, "D")
print("B: inicio del arco ciego (primeros 10 objetos):", {f"obj{k}": v for k, v in list(spB["arc_starts"].items())[:10]})
print("D: objetos desconocidos (semilla 42):", [f"obj{o}" for o in spD["unknown_objects"]])

,protocolo,train,test,fit (val),val,fuga train∩test
0,A,3600,3600,1800,1800,0
1,B,6000,1200,4800,1200,0
2,C,1200,6000,600,600,0
3,D,3240,3960,1440,1800,0


B: inicio del arco ciego (primeros 10 objetos): {'obj1': 30, 'obj2': 275, 'obj3': 235, 'obj4': 155, 'obj5': 155, 'obj6': 305, 'obj7': 30, 'obj8': 250, 'obj9': 70, 'obj10': 30}
D: objetos desconocidos (semilla 42): ['obj9', 'obj10', 'obj20', 'obj42', 'obj61', 'obj69', 'obj72', 'obj83', 'obj95', 'obj97']


## 4. Experimentación

**Características clásicas** (`features.py`, ~3 ms/imagen): histograma HSV 16×4×4 sobre la máscara de objeto
(raíz cuadrada → Hellinger), HOG 64×64 (9 orientaciones, celdas 8×8), LBP uniforme en rejilla 2×2 y momentos
de Hu; estandarización + PCA(150) ajustados **solo con train**.

**Clasificadores comparados** (rejilla pequeña, elegida en validación): kNN (k∈{1,5}), regresión logística
(C∈{0.1,1,10}), SVM RBF con probabilidades de Platt (C∈{1,10}) y Random Forest (300 árboles).

**Ángulo:** recuperación condicionada al objeto — entre las vistas de referencia de la clase predicha se toma
la más cercana y se refina con la media circular ponderada de los 2 vecinos (interpolación); la
**incertidumbre** (en grados) se obtiene invirtiendo la curva distancia-vs-separación del propio objeto: "la
distancia del query a su vecino más cercano equivale a X° de giro". La misma recuperación da el top-k similar.

**Desconocido / dudoso:** distancia al vecino más cercano (novedad) → REJECT; confianza de clase baja,
incertidumbre angular alta o ambigüedad rotacional → REVIEW. Umbrales elegidos en validación para maximizar
la fracción automatizada con **error entre ACT ≤ 2 %**; si ningún umbral lo logra, la automatización se
desactiva (todo a REVIEW).

**Comparaciones obligatorias:** línea base 1-NN en píxeles crudos 32×32 (ángulo = el del vecino, sin rechazo)
y referencia moderna: **ResNet18 ImageNet congelada** (embedding 512-d) con exactamente el mismo pipeline
(clasificadores, ángulo, umbrales) para aislar el efecto de la representación.

In [7]:
OUT_DIR = ROOT / "data" / "runtime" / "experiments"
t = time.time()
results = vexp.run_all(data, features=("classic", "resnet18"), out_dir=OUT_DIR, log=lambda *a: None)
print(f"experimentos: {time.time() - t:.0f}s")

2026/10/24 12:25:59 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/nico/Documents/IA/.venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


experimentos: 298s


### 4.1 Validación y selección de modelo

In [8]:
vrows = []
for p, rr in results["protocols"].items():
    for k, m in rr["classic"]["validation"].items():
        clf, params = k.split(":", 1)
        vrows.append({"protocolo": p, "clasificador": clf, "params": params, "val_exactitud": m["class_accuracy"],
                      "val_e2e": m["end_to_end_success"], "fit_s": m["fit_s"]})
vdf = pd.DataFrame(vrows)
display(vdf.pivot_table(index=["clasificador", "params"], columns="protocolo", values="val_exactitud").round(4))
prod = vexp.select_production_classifier(results, "classic")
print("Seleccionado para producción (mejor exactitud media de VALIDACIÓN en A–D):", prod["clf"], prod["params"])
display(pd.DataFrame({k: v for k, v in prod["mean_val"].items() if not k.startswith("_")}).T.round(4))

protocolo                                 A       B       C       D
clasificador  params                                               
knn           {"n_neighbors": 1}     0.9772  0.8817  0.7817  0.9757
              {"n_neighbors": 5}     0.9683  0.8433  0.7467  0.9688
logreg        {"C": 0.1}             0.9933  0.9517  0.8783  0.9917
              {"C": 1.0}             0.9917  0.9542  0.8817  0.9917
              {"C": 10.0}            0.9917  0.9467  0.8800  0.9917
random_forest {"n_estimators": 300}  0.9689  0.8842  0.8050  0.9681
svm_rbf       {"C": 1.0}             0.9761  0.9292  0.7650  0.9701
              {"C": 10.0}            0.9928  0.9342  0.7850  0.9903

Seleccionado para producción (mejor exactitud media de VALIDACIÓN en A–D): logreg {'C': 1.0}


,mean_val_class_accuracy,mean_val_end_to_end
knn,0.9041,0.4999
logreg,0.9552,0.5061
svm_rbf,0.9256,0.5058
random_forest,0.9065,0.4948


La selección usa solo validación. Regresión logística y SVM RBF quedan prácticamente empatadas en exactitud,
pero la logística está **mucho mejor calibrada** (ver ECE abajo), es ~10× más rápida de entrenar y su artefacto
es pequeño; kNN y Random Forest quedan detrás en los protocolos con brecha (B, C).

### 4.2 Resultados en test por protocolo (evaluación única)

In [9]:
def table(feature):
    rows = []
    for p, rr in results["protocols"].items():
        r = rr[feature]
        for clf, m in r["test"].items():
            rows.append({"protocolo": p, "modelo": f"{feature}:{clf}" + (" *" if clf == r["selected"]["clf"] else ""),
                         "exactitud": m["class_accuracy"], "MAE ángulo°": m["angle_mae_deg"], "≤10°": m["angle_within_10"],
                         "éxito e2e": m["end_to_end_success"], "ACT": m["coverage_act"], "REVIEW": m["review_rate"],
                         "REJECT": m["reject_rate"], "error|ACT": m["error_among_act"], "ECE": m["ece"],
                         "AUROC D": m.get("auroc_combined_rank", np.nan)})
        if feature == "classic":
            b = r["baseline_raw_1nn"]
            rows.append({"protocolo": p, "modelo": "baseline: píxeles 1-NN", "exactitud": b["class_accuracy"],
                         "MAE ángulo°": b["angle_mae_deg"], "≤10°": b["angle_within_10"], "éxito e2e": b["end_to_end_success"],
                         "ACT": 1.0, "REVIEW": 0.0, "REJECT": 0.0, "error|ACT": b["error_among_act"], "ECE": np.nan,
                         "AUROC D": b.get("auroc_nn_distance", np.nan)})
    return pd.DataFrame(rows)
res_classic = table("classic"); res_resnet = table("resnet18")
display(res_classic.round(4)); display(res_resnet.round(4))
fig = va.per_protocol_bars(results); va.savefig(fig, "03_per_protocol"); plt.show()

,protocolo,modelo,exactitud,MAE ángulo°,≤10°,éxito e2e,ACT,REVIEW,REJECT,error|ACT,ECE,AUROC D
0,A,classic:knn,0.9978,2.6609,0.9811,0.9789,0.9300,0.0700,0.0000,0.0030,0.0022,NaN
1,A,classic:logreg *,0.9997,2.7068,0.9808,0.9806,0.9214,0.0786,0.0000,0.0021,0.0143,NaN
2,A,classic:svm_rbf,1.0000,2.7066,0.9808,0.9808,0.9217,0.0783,0.0000,0.0021,0.5385,NaN
3,A,classic:random_forest,0.9986,2.7074,0.9808,0.9794,0.9039,0.0961,0.0000,0.0025,0.3547,NaN
4,A,baseline: píxeles 1-NN,0.9997,5.9280,0.9761,0.9758,1.0000,0.0000,0.0000,0.0242,NaN,NaN
5,B,classic:knn,0.9167,64.3067,0.1791,0.1642,0.0025,0.9950,0.0025,0.6667,0.0833,NaN
6,B,classic:logreg *,0.9700,67.4719,0.1692,0.1642,0.0000,0.9975,0.0025,0.0000,0.0177,NaN
7,B,classic:svm_rbf,0.9617,66.2567,0.1707,0.1642,0.0000,0.9975,0.0025,0.0000,0.4987,NaN
8,B,classic:random_forest,0.9300,66.0672,0.1765,0.1642,0.0025,0.9950,0.0025,0.6667,0.4644,NaN
9,B,baseline: píxeles 1-NN,0.9292,61.2242,0.3076,0.2858,1.0000,0.0000,0.0000,0.7142,NaN,NaN


,protocolo,modelo,exactitud,MAE ángulo°,≤10°,éxito e2e,ACT,REVIEW,REJECT,error|ACT,ECE,AUROC D
0,A,resnet18:knn *,1.0000,1.9826,0.9878,0.9878,0.7178,0.2822,0.0000,0.0031,0.0000,NaN
1,A,resnet18:logreg,1.0000,1.9826,0.9878,0.9878,0.7172,0.2828,0.0000,0.0031,0.0035,NaN
2,A,resnet18:svm_rbf,1.0000,1.9826,0.9878,0.9878,0.7178,0.2822,0.0000,0.0031,0.6570,NaN
3,A,resnet18:random_forest,0.9997,1.9826,0.9878,0.9875,0.7175,0.2825,0.0000,0.0031,0.1931,NaN
4,B,resnet18:knn,0.9900,48.7034,0.1599,0.1583,0.0000,0.9875,0.0125,0.0000,0.0100,NaN
5,B,resnet18:logreg *,0.9967,49.4053,0.1589,0.1583,0.0000,0.9875,0.0125,0.0000,0.0154,NaN
6,B,resnet18:svm_rbf,0.9967,49.4053,0.1589,0.1583,0.0000,0.9875,0.0125,0.0000,0.5640,NaN
7,B,resnet18:random_forest,0.9808,48.9184,0.1614,0.1583,0.0000,0.9875,0.0125,0.0000,0.2850,NaN
8,C,resnet18:knn,0.9977,7.6454,0.8745,0.8725,0.0000,1.0000,0.0000,0.0000,0.0287,NaN
9,C,resnet18:logreg *,0.9995,7.6759,0.8738,0.8733,0.0000,1.0000,0.0000,0.0000,0.0523,NaN


`*` = clasificador elegido en validación para ese protocolo. *éxito e2e* = clase correcta **y** error angular
circular ≤ 10° (sobre todas las vistas de test de objetos conocidos); *ACT* = fracción automatizada;
*error|ACT* = fracción de decisiones ACT que habrían sido un empaque equivocado o un agarre fallido (en D incluye
desconocidos aceptados como conocidos). AUROC D = combinación por rangos de distancia y confianza.

### 4.3 Ángulo: error circular y alternativas

In [10]:
outs = {p: va.load_outputs("classic", p, OUT_DIR) for p in "ABCD"}
fig = va.angle_error_hist({p: outs[p] for p in "ABC"}); va.savefig(fig, "04_angle_error_hist"); plt.show()
# alternativa: sin interpolación (ángulo del 1-NN) vs interpolación con 2 y 3 vecinos — mismo clasificador
from src.vision.experiments import _fit_predict
from src.vision.angle import circular_error
names, X = data["names"], data["classic"]
arows = []
for p in "ABC":
    sp = protocol_split(ds.labels, ds.angles, p); tr, te = sp["train"], sp["test"]
    for k in (1, 2, 3):
        cfg = {"clf": "logreg", "params": prod["params"], "k_angle_refine": k}
        _, o, _, _ = _fit_predict(X[tr], names[tr], ds.angles[tr], ds.image_ids[tr], X[te], cfg)
        ok = o["label"] == names[te]; e = circular_error(o["angle"], ds.angles[te])[ok]
        arows.append({"protocolo": p, "vecinos interpolados": k, "MAE°": e.mean(), "mediana°": np.median(e), "≤10°": (e <= 10).mean()})
display(pd.DataFrame(arows).pivot_table(index="vecinos interpolados", columns="protocolo", values=["MAE°", "≤10°"]).round(3))

MAE°                   ≤10°              
protocolo                 A       B       C      A      B      C
vecinos interpolados                                            
1                     6.069  66.469  13.128  0.982  0.286  0.705
2                     2.707  67.472   9.394  0.981  0.169  0.899
3                     3.128  68.572  10.449  0.983  0.158  0.858

**Lectura del ángulo.** La métrica es el error circular `min(|a−b|, 360−|a−b|)`; reportamos MAE, mediana y
% ≤ 10° (la tolerancia de agarre). La interpolación con 2 vecinos reduce el MAE frente a copiar el ángulo del
vecino más cercano (el test cae *entre* vistas de referencia). En **B** (arco ciego de 60°) ningún método
basado en apariencia puede "inventar" las vistas no vistas: solo las vistas en los bordes del arco se recuperan
bien; el resto debe ir a REVIEW. En **C** (12 vistas) el ángulo es útil pero no alcanza la confiabilidad
exigida para automatizar.

### 4.4 Confianza: calibración, riesgo–cobertura y rechazo de desconocidos

In [11]:
fig = va.reliability_and_risk({p: outs[p] for p in "ABCD"}); va.savefig(fig, "05_reliability_risk"); plt.show()
fig = va.unknown_roc(outs["D"]); va.savefig(fig, "06_unknown_roc"); plt.show()
cal = []
for feat in ("classic", "resnet18"):
    for p, rr in results["protocols"].items():
        for clf, m in rr[feat]["test"].items():
            cal.append({"rep": feat, "protocolo": p, "clf": clf, "ECE": m["ece"], "AURC(conf)": m["aurc_confidence"]})
display(pd.DataFrame(cal).pivot_table(index=["rep", "clf"], columns="protocolo", values="ECE").round(4))
d = results["protocols"]["D"]["classic"]; m = d["test"][d["selected"]["clf"]]
print({k: round(m[k], 4) for k in ("auroc_nn_distance", "auroc_neg_confidence", "auroc_combined_rank", "unknown_rejected",
                                   "unknown_false_accept_act", "unknown_not_act", "known_false_reject", "known_coverage_act")})
print("umbrales D (validación):", {k: round(v, 3) if isinstance(v, float) else v for k, v in d["thresholds"][d["selected"]["clf"]].items()})

protocolo                    A       B       C       D
rep      clf                                          
classic  knn            0.0022  0.0833  0.0222  0.0028
         logreg         0.0143  0.0177  0.0181  0.0005
         random_forest  0.3547  0.4644  0.5149  0.3522
         svm_rbf        0.5385  0.4987  0.7359  0.5109
resnet18 knn            0.0000  0.0100  0.0287  0.0000
         logreg         0.0035  0.0154  0.0523  0.0187
         random_forest  0.1931  0.2850  0.3717  0.1853
         svm_rbf        0.6570  0.5640  0.8408  0.6300

{'auroc_nn_distance': 0.9799, 'auroc_neg_confidence': 0.989, 'auroc_combined_rank': 0.991, 'unknown_rejected': 0.7125, 'unknown_false_accept_act': 0.0514, 'unknown_not_act': 0.9486, 'known_false_reject': 0.0151, 'known_coverage_act': 0.921}
umbrales D (validación): {'min_class_confidence': 0.95, 'max_angle_uncertainty_deg': 30.0, 'unknown_distance': 24.486, 'val_coverage': 0.606, 'val_error_among_act': 0.017, 'target_error': 0.02, 'angle_threshold_source': 'validation', 'automation_enabled': True}


**¿Es la confianza una base válida para decisiones automáticas?**

1. **Calibración:** la regresión logística tiene ECE bajo (la confianza ≈ probabilidad real de acierto); el SVM
   con Platt multiclase y el Random Forest están mal calibrados (ECE alto) aunque acierten igual → no deben usarse
   sus probabilidades crudas para decidir. Esto es un argumento adicional para la logística.
2. **Riesgo–cobertura:** ordenar por confianza (o por incertidumbre angular) reduce monótonamente el error entre
   los casos automatizados → existe un umbral operable. La incertidumbre angular es la señal que más importa en
   A/C (el error de clase casi no existe); la curva "P(|error| ≤ 10°) vs incertidumbre" muestra que la
   incertidumbre en grados es informativa y coherente con la tolerancia.
3. **Desconocidos (D):** la distancia al vecino más cercano y la confianza separan conocidos de desconocidos con
   AUROC alto; el umbral (Youden en validación con pseudo-desconocidos) rechaza la mayoría de desconocidos y casi
   ninguno termina en ACT.

### 4.5 Criterio de aceptación: tolerancia ±10° → métrica de negocio

La tolerancia de agarre de ±10° se traduce en: **éxito extremo a extremo = clase correcta ∧ error circular ≤ 10°**.
El robot solo actúa en ACT; por eso la métrica de negocio es el par (**fracción automatizada**, **error entre ACT**
= tasa de empaque equivocado o agarre fallido entre lo que el robot ejecuta). El umbral de incertidumbre angular
se elige para que el error entre ACT ≤ 2 % en validación.

In [12]:
acc = []
for p, rr in results["protocols"].items():
    for feat, lab in (("classic", "Clásico (prod.)"), ("resnet18", "ResNet18")):
        r = rr[feat]; m = r["test"][r["selected"]["clf"]]; th = r["thresholds"][r["selected"]["clf"]]
        acc.append({"protocolo": p, "modelo": lab, "éxito e2e (sin rechazo)": m["end_to_end_success"], "% ACT": m["coverage_act"],
                    "error|ACT": m["error_among_act"], "éxito automatizado": m["automated_success_rate"],
                    "umbral incert.°": th["max_angle_uncertainty_deg"], "umbral conf.": th["min_class_confidence"],
                    "automatización habilitada": th.get("automation_enabled", True)})
    b = rr["classic"]["baseline_raw_1nn"]
    acc.append({"protocolo": p, "modelo": "Base píxeles 1-NN", "éxito e2e (sin rechazo)": b["end_to_end_success"], "% ACT": 1.0,
                "error|ACT": b["error_among_act"], "éxito automatizado": b["end_to_end_success"]})
acc_df = pd.DataFrame(acc); display(acc_df.round(4))

,protocolo,modelo,éxito e2e (sin rechazo),% ACT,error|ACT,éxito automatizado,umbral incert.°,umbral conf.,automatización habilitada
0,A,Clásico (prod.),0.9806,0.9214,0.0021,0.9194,30.0,0.30,True
1,A,ResNet18,0.9878,0.7178,0.0031,0.7156,7.5,0.99,True
2,A,Base píxeles 1-NN,0.9758,1.0000,0.0242,0.9758,NaN,NaN,NaN
3,B,Clásico (prod.),0.1642,0.0000,0.0000,0.0000,5.0,0.60,True
4,B,ResNet18,0.1583,0.0000,0.0000,0.0000,5.0,0.60,True
5,B,Base píxeles 1-NN,0.2858,1.0000,0.7142,0.2858,NaN,NaN,NaN
6,C,Clásico (prod.),0.8908,0.0003,0.0000,0.0003,5.0,0.70,True
7,C,ResNet18,0.8733,0.0000,0.0000,0.0000,5.0,0.30,True
8,C,Base píxeles 1-NN,0.7428,1.0000,0.2572,0.7428,NaN,NaN,NaN
9,D,Clásico (prod.),0.9846,0.7629,0.0146,0.7518,30.0,0.95,True


## 5. Análisis de errores

In [13]:
fails = va.per_object_failures({p: outs[p] for p in "ABC"}, top=8)
for p, rows in fails.items():
    print(f"Protocolo {p}: objetos con más fallos e2e"); display(pd.DataFrame(rows))
for p in ("B", "C"):
    fig = va.failure_examples_figure(ds, outs, p); va.savefig(fig, f"07_failures_{p}"); plt.show()
# ¿los fallos se concentran en objetos ambiguos?
fA = outs["A"]; okA = (fA["label"] == fA["true_label"]) & (circular_error(fA["angle"], fA["true_angle"]) <= 10)
per_obj = pd.Series(~okA).groupby(fA["true_label"]).mean()
j = symdf.join(per_obj.rename("tasa_fallo_A"))
print("Correlación de Spearman índice de perfil vs. tasa de fallo en A:", round(j[["perfil", "tasa_fallo_A"]].corr("spearman").iloc[0, 1], 3))

Protocolo A: objetos con más fallos e2e


,object,failures,n,class_errors,confused_with
0,obj95,17,36,0,None
1,obj47,14,36,0,None
2,obj94,12,36,0,None
3,obj98,7,36,0,None
4,obj67,5,36,0,None
5,obj35,3,36,0,None
6,obj84,3,36,0,None
7,obj13,1,36,0,None


Protocolo B: objetos con más fallos e2e


,object,failures,n,class_errors,confused_with
0,obj98,12,12,0,None
1,obj95,12,12,0,None
2,obj84,12,12,0,None
3,obj96,11,12,0,None
4,obj94,11,12,0,None
5,obj47,11,12,0,None
6,obj69,11,12,0,None
7,obj67,11,12,0,None


Protocolo C: objetos con más fallos e2e


,object,failures,n,class_errors,confused_with
0,obj94,31,60,0,NaN
1,obj95,29,60,0,NaN
2,obj84,25,60,10,obj59
3,obj47,24,60,0,NaN
4,obj67,20,60,7,obj37
5,obj98,20,60,2,obj75
6,obj35,16,60,0,NaN
7,obj21,13,60,5,obj14


Correlación de Spearman índice de perfil vs. tasa de fallo en A: 0.19


**Lectura.** Los fallos angulares en A/C/D se concentran en pocos objetos: los de mayor índice de perfil
(donas y taza: obj95, obj94, obj47) y los de mayor índice LOO (vistas "gemelas" a 180°: obj98, obj67, obj84). La
correlación de rangos sobre los 100 objetos es moderada porque la mayoría tiene 0 fallos; la concentración en la
cola es lo relevante. El error en esos casos es de ±180° o "cualquier ángulo", no de unos pocos grados, y son los
que el sistema marca como `rotational_ambiguity` / `high_angle_uncertainty` → REVIEW. Los errores de clase se dan entre objetos de la misma familia visual (pares
casi duplicados de la sección 2). En B el fallo dominante es estructural (arco no visto).

## 6. Costo: latencia y tamaño del artefacto (clásico vs. referencia moderna)

In [14]:
import subprocess
r = subprocess.run([sys.executable, str(ROOT / "scripts" / "train_vision.py")], capture_output=True, text=True, cwd=ROOT)
print(r.stdout[-1500:]); print(r.stderr[-800:] if r.returncode else "")

Clasificador: logreg {'C': 1.0} (fuente: docs/report/results.json (validación))
Clases con ángulo poco fiable en validación: ['obj47', 'obj94', 'obj95']
Umbrales (validación): {'min_class_confidence': 0.99, 'max_angle_uncertainty_deg': 30.0, 'unknown_distance': 19.47063943862915} {'val_coverage': 0.7325757575757575, 'val_error_among_act': 0.0079283005860048, 'target_error': 0.02, 'angle_threshold_source': 'validation', 'automation_enabled': True}
Entrenamiento final: 1.0s; clases simétricas: ['obj47', 'obj67', 'obj84', 'obj94', 'obj95', 'obj98']
Artefacto exportado en /Users/nico/Documents/IA/models/vision-1.0.0 (9.4 MB)




*(La celda anterior exporta el artefacto productivo; ver sección 7. Aquí se usa para medir la latencia real del
pipeline completo: calidad + características + clasificador + ángulo + top-k + decisión.)*

In [15]:
from src.vision.inference import VisionPredictor
pred = VisionPredictor.load(ROOT / "models" / "vision-1.0.0")
rng = np.random.default_rng(42); sample = ds.images[rng.choice(len(ds.images), 100, replace=False)]
lat = {"classic (pipeline completo)": va.measure_latency(pred, sample)}
t = time.perf_counter(); _ = vexp.raw_pixels(sample[:1], 32); lat["baseline píxeles 1-NN*"] = {"p50_ms": results["protocols"]["A"]["classic"]["baseline_raw_1nn"]["predict_ms_per_image_batch"]}
rl = va.measure_resnet_latency(sample, n=50)
lat["resnet18 CPU (solo embedding)"] = rl["cpu"]
if "mps" in rl: lat["resnet18 MPS (solo embedding)"] = rl["mps"]
display(pd.DataFrame(lat).T.round(2))
size_mb = sum(f.stat().st_size for f in (ROOT / "models" / "vision-1.0.0").glob("*") if f.is_file()) / 1e6
print(f"Artefacto clásico: {size_mb:.1f} MB | pesos ResNet18 fp32: {rl['weights_mb_fp32']} MB (+ torch/torchvision ~ cientos de MB)")
fig = va.latency_figure({k: v for k, v in lat.items()}); va.savefig(fig, "08_latency"); plt.show()
COST = {"latency": lat, "classic_artifact_mb": size_mb, "resnet18_weights_mb": rl["weights_mb_fp32"]}

,p50_ms,p95_ms,mean_ms
classic (pipeline completo),14.84,36.10,18.85
baseline píxeles 1-NN*,0.56,NaN,NaN
resnet18 CPU (solo embedding),29.91,33.48,NaN
resnet18 MPS (solo embedding),16.58,20.61,NaN


Artefacto clásico: 9.4 MB | pesos ResNet18 fp32: 44.7 MB (+ torch/torchvision ~ cientos de MB)


**¿Justifica la brecha el costo?** La referencia moderna mejora sobre todo la exactitud de clase en los
protocolos con brecha angular (B, C) y el MAE de ángulo, pero (a) **no resuelve** el problema central del arco
ciego (B) ni habilita automatizar C; (b) en A/D —el régimen operativo real, con referencias densas— el pipeline
clásico ya está cerca del techo: con la decisión ACT/REVIEW automatiza más casos que la ResNet, con error entre
ACT bajo (la ResNet logra menor error entre ACT en D, a costa de automatizar menos; ver tabla 4.5); (c) solo el
embedding de ResNet18 ya cuesta más en CPU que el pipeline clásico completo, añade ~45 MB de pesos más la
dependencia de PyTorch, y es menos interpretable. Conclusión: **producción =
clásico**; ResNet18 queda como referencia y como candidata si en planta (con fondos/iluminación reales) la brecha
crece — decisión a tomar con datos del piloto.

## 7. Exportación del artefacto y registro

In [16]:
results["production_selection"] = {"clf": prod["clf"], "params": prod["params"], "criterion": "max exactitud media de validación A–D"}
results["production_selection_scores"] = prod["mean_val"]
results["summary"] = va.build_summary(results)
results["cost"] = COST
results["symmetry"] = {"profile_score": {f"obj{k}": v for k, v in sym["profile_score"].items()},
                       "loo_score": {f"obj{k}": v for k, v in sym["loo_score"].items()}}
results["confusable_pairs"] = pairs
results["error_analysis"] = fails
results["angle_alternatives"] = arows
results["acceptance_table"] = acc_df.to_dict(orient="records")
p = va.write_results(vexp.to_jsonable(results)); print("escrito", p)
# re-exporta con la selección registrada en results.json (train_vision.py la lee de ahí)
r = subprocess.run([sys.executable, str(ROOT / "scripts" / "train_vision.py")], capture_output=True, text=True, cwd=ROOT)
print(r.stdout[-1200:])
meta = json.loads((ROOT / "models" / "vision-1.0.0" / "metadata.json").read_text())
print({k: meta[k] for k in ("version", "classifier", "thresholds", "symmetric_classes", "artifact_size_mb", "data_version_hash")})
print(f"Tiempo total del notebook: {(time.time() - T0) / 60:.1f} min")

escrito /Users/nico/Documents/IA/docs/report/results.json


Clasificador: logreg {'C': 1.0} (fuente: docs/report/results.json (validación))
Clases con ángulo poco fiable en validación: ['obj47', 'obj94', 'obj95']
Umbrales (validación): {'min_class_confidence': 0.99, 'max_angle_uncertainty_deg': 30.0, 'unknown_distance': 19.47063943862915} {'val_coverage': 0.7325757575757575, 'val_error_among_act': 0.0079283005860048, 'target_error': 0.02, 'angle_threshold_source': 'validation', 'automation_enabled': True}
Entrenamiento final: 1.1s; clases simétricas: ['obj47', 'obj67', 'obj84', 'obj94', 'obj95', 'obj98']
Artefacto exportado en /Users/nico/Documents/IA/models/vision-1.0.0 (9.4 MB)

{'version': 'vision-1.0.0', 'classifier': {'name': 'logreg', 'params': {'C': 1.0}, 'selection_source': 'docs/report/results.json (validación)'}, 'thresholds': {'min_class_confidence': 0.99, 'max_angle_uncertainty_deg': 30.0, 'unknown_distance': 19.47063943862915}, 'symmetric_classes': ['obj47', 'obj67', 'obj84', 'obj94', 'obj95', 'obj98'], 'artifact_size_mb': 9.4, 'da

## 8. Conclusiones

* **Modelo productivo:** características clásicas (HSV + HOG + LBP + Hu) → PCA(150) → regresión logística, con
  ángulo por recuperación condicionada al objeto e incertidumbre en grados; decisión ACT/REVIEW/REJECT con umbrales
  fijados en validación. Artefacto pequeño, ~15 ms/imagen en CPU, trazable (hash de datos, MLflow, metadata).
* **Protocolos:** A y D son automatizables (ver tabla 4.5: fracción ACT alta y error entre ACT cercano al
  objetivo de 2 %); C (12 vistas/objeto) y B (arco ciego) **no** lo son con la confiabilidad exigida — el sistema
  lo detecta en validación y envía a REVIEW (o desactiva la automatización) en lugar de actuar. Esto se
  traduce en el requisito operativo para dar de alta un SKU: **≥ 12 vistas cubriendo ≥ 300°** como mínimo para
  staging, y referencias densas (idealmente cada 5–10°) para habilitar ACT.
* **Limitaciones:** ver la model card (`docs/model_card.md`) — dataset de laboratorio, un solo eje de rotación,
  sin oclusión ni variación de fondo/iluminación; requiere piloto en sombra y recalibración en planta.